In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import matplotlib.pyplot as plt
import seaborn as sns


## 1. Load data

In [ ]:
MODEL_NAME = "google/muril-large-cased"
MAX_LEN = 96
SEED = 42
CONF_THRESHOLD = 0.55

# 1.Load data

DATA_DIR = "/home/shivmexe/Projects/grievance_sih/ml/data/grievance_classifier" 
train_pool = pd.read_csv(f"{DATA_DIR}/training_data.csv")
hard_holdout = pd.read_csv(f"{DATA_DIR}/holdout_data.csv")

le = LabelEncoder()
train_pool["label"] = le.fit_transform(train_pool["category"])
hard_holdout["label"] = le.transform(hard_holdout["category"])

num_labels = len(le.classes_)
print(f"{num_labels} categories: {list(le.classes_)}")
print(train_pool["category"].value_counts())


## 2. Train/val split

In [ ]:
train_df, val_df = train_test_split(
    train_pool, test_size=0.15, random_state=SEED, stratify=train_pool["label"]
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_labels),
    y=train_df["label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print(dict(zip(le.classes_, class_weights.tolist())))

train_ds = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[["text", "label"]].reset_index(drop=True))
holdout_ds = Dataset.from_pandas(hard_holdout[["text", "label"]].reset_index(drop=True))


## 3. Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
holdout_ds = holdout_ds.map(tokenize, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


## 4. Model: MuRIL-large + LoRA 

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "key", "value", "dense"],
    modules_to_save=["classifier"],   
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()   


## 5. Trainer

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, label_smoothing=0.05, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.label_smoothing = label_smoothing

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = nn.CrossEntropyLoss(weight=weight, label_smoothing=self.label_smoothing)
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

effective_batch = 8 * 4  
steps_per_epoch = max(1, len(train_ds) // effective_batch)
total_steps = steps_per_epoch * 8  
warmup_steps = int(0.1 * total_steps)

args = TrainingArguments(
    output_dir="./grievance_muril_large_lora",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,          
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,   
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    label_smoothing=0.05,
)


## 6. Training

In [ ]:
trainer.train()

## 7. Evaluation 

In [ ]:
print("\n=== Validation set ===")
print(trainer.evaluate(val_ds))

print("\n=== Holdout set ===")
holdout_metrics = trainer.evaluate(holdout_ds)
print(holdout_metrics)

preds = trainer.predict(holdout_ds)
pred_labels = np.argmax(preds.predictions, axis=-1)
print(classification_report(
    hard_holdout["label"], pred_labels, target_names=le.classes_, digits=3
))

cm = confusion_matrix(hard_holdout["label"], pred_labels)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.savefig("confusion_matrix_large_lora.png", dpi=150)
plt.show()



## 8. Saving Model

In [ ]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained("home/shivmexe/Projects/grievance_sih/ml/models/large_model")
tokenizer.save_pretrained("home/shivmexe/Projects/grievance_sih/ml/models/large_model")

import json
with open("home/shivmexe/Projects/grievance_sih/ml/models/large_model/label_map.json", "w", encoding="utf-8") as f:
    json.dump({str(i): c for i, c in enumerate(le.classes_)}, f, ensure_ascii=False, indent=2)

print("Save model ")

## 9. Test

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch, json

with open("home/shivmexe/Projects/grievance_sih/ml/models/large_model/label_map.json", "r", encoding="utf-8") as f:
    label_map = json.load(f)
id2label = {int(k): v for k, v in label_map.items()}

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "cpu"
infer_tokenizer = AutoTokenizer.from_pretrained("./large_model_final")
infer_model = AutoModelForSequenceClassification.from_pretrained("./large_model_final").to(device)
infer_model.eval()

def predict_grievance(text, threshold=CONF_THRESHOLD):
    inputs = infer_tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN, padding=True).to(device)
    with torch.no_grad():
        logits = infer_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
        pred_id = probs.argmax(dim=-1).item()
        confidence = probs[0][pred_id].item()
    label = id2label[pred_id]
    if confidence < threshold:
        return "NEEDS_MANUAL_REVIEW", confidence, label
    return label, confidence, label

new_complaints = [
    "Sir, pichle 10 din se hamare mohalle ka streetlight band hai, raat ko chori hone ka dar hai.",
    "Mera ration card 1 mahine se pending hai, dealer anaj nahi de raha.",
    "विद्यालय में छात्रवृत्ति का पैसा अभी तक नहीं आया है।",
    "Sadak par aawara kutte bahut hain, kal ek bachche ko kaat liya.",
    "Mera mobile network 2 hafte se kaam nahi kar raha is area mein.",
    "sadak ke ghade ho rhe he mere yaha",
    "mere vidyalay me shikshoka ki kami he",
    "mere yaha gunda gardi bad rhi he",
    "bijli kab aayegi",
    "aaj kal baarish nhi ho rhi he",
    "mere khate me se 500 rupaye gayad ho gye",
]

print("\n--- Predictions ---")
for complaint in new_complaints:
    route, confidence, best_guess = predict_grievance(complaint)
    print(f"Text: {complaint}")
    if route == "NEEDS_MANUAL_REVIEW":
        print(f"-> Low confidence ({confidence*100:.1f}%), best guess was '{best_guess}'. Routing to manual review.\n")
    else:
        print(f"Predicted Category: {route} (Confidence: {confidence*100:.2f}%)\n")
